In [1]:
# 单元格 1：导入依赖与配置
import json
import sys
from pathlib import Path
import re

sys.path.append(str(Path.cwd()))

from config import NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, ENTITY_LABELS, RELATION_TYPES, SIMILARITY_THRESHOLD
from kg_client import Neo4jClient
from utils import ollama_chat, ollama_embed, cosine_similarity, triple_to_text
from prompts import EXTRACT_GLOBAL_PROMPT, fill_prompt

print("配置加载完成。")

配置加载完成。


In [2]:
# 单元格 2：初始化 Neo4j 客户端
client = Neo4jClient()
try:
    result = client.query("MATCH (n) RETURN count(n) AS total")
    total_nodes = result[0]['total'] if result else 0
    print(f"Neo4j 连接成功！节点总数：{total_nodes}")
except Exception as e:
    print(f"连接失败：{e}")
    client = None

Neo4j 连接成功！节点总数：1212


In [3]:
# 单元格 3：加载思维导图的叶子问题列表
mind_map_file = "data/mind_map_sample.json"
leaf_questions = []

if Path(mind_map_file).exists():
    with open(mind_map_file, "r", encoding="utf-8") as f:
        mind_map_tree = json.load(f)
    def collect_leaf_questions(nodes):
        leafs = []
        for node in nodes:
            if not node.get("children"):
                leafs.append(node["sub_question"])
            else:
                leafs.extend(collect_leaf_questions(node["children"]))
        return leafs
    leaf_questions = collect_leaf_questions(mind_map_tree)
    print(f"已从文件加载 {len(leaf_questions)} 个叶子问题。")
else:
    leaf_questions = [
        "华佗五禽戏模仿了哪五种动物？",
        "华佗五禽戏的代表传承人有哪些？"
    ]
    print("未找到思维导图文件，使用手动问题列表。")

print("\n叶子问题列表：")
for i, q in enumerate(leaf_questions, 1):
    print(f"{i}. {q}")

已从文件加载 2 个叶子问题。

叶子问题列表：
1. 陈氏太极拳的动作要素有哪些？
2. 陈氏太极拳的代表传承人有哪些？


In [4]:
# 单元格 4：定义全局子图提取函数
def extract_global_subgraph(question_list):
    prompt = fill_prompt(EXTRACT_GLOBAL_PROMPT, mind_map=json.dumps(question_list, ensure_ascii=False))
    print("正在调用 LLM 提取全局子图...")
    response = ollama_chat(prompt, temperature=0.1, format_json=False)
    
    print(f"[调试] 原始响应长度：{len(response)} 字符")
    print(f"[调试] 原始响应前300字符：{repr(response[:300])}")
    
    array_match = re.search(r'\[.*\]', response, re.DOTALL)
    if not array_match:
        print("未找到 JSON 数组，返回空列表")
        return []
    
    try:
        triples = json.loads(array_match.group(0))
        if isinstance(triples, list):
            valid_triples = []
            for t in triples:
                if isinstance(t, list) and len(t) == 3:
                    valid_triples.append(t)
            return valid_triples
        else:
            return []
    except Exception as e:
        print(f"解析失败：{e}")
        return []

In [5]:
# 单元格 5：执行子图提取
global_triple_patterns = extract_global_subgraph(leaf_questions)

print(f"\n提取到的全局子图模式（共 {len(global_triple_patterns)} 个三元组）：")
for i, (s, r, o) in enumerate(global_triple_patterns, 1):
    print(f"{i}. ({s}) -[{r}]-> ({o})")

正在调用 LLM 提取全局子图...
[调试] 原始响应长度：49 字符
[调试] 原始响应前300字符：'[["陈氏太极拳", "展现", "动作要素"], ["陈氏太极拳", "传承", "传承人"]]'

提取到的全局子图模式（共 2 个三元组）：
1. (陈氏太极拳) -[展现]-> (动作要素)
2. (陈氏太极拳) -[传承]-> (传承人)


In [6]:
# 单元格 6：定义子图匹配函数（带参数控制）
# ========== 可调参数 ==========
GLOBAL_TOP_K_NODES = 2   # 全局匹配时每个模式尝试的主体节点数量
# =============================

def match_global_subgraph(triple_patterns, client, max_results_per_pattern=5, top_k_nodes=GLOBAL_TOP_K_NODES):
    all_triples = set()
    
    for start, rel, end in triple_patterns:
        # 1. 精确匹配三元组
        exact = client.match_triple(start, rel, end)
        if exact:
            print(f"  模式 ({start})-[:{rel}]->({end}) 精确匹配到 {len(exact)} 条")
            all_triples.update(exact)
            continue
        
        # 2. 宽松匹配：主体+关系，客体不限
        print(f"  模式 ({start})-[:{rel}]->({end}) 精确匹配失败，尝试宽松匹配...")
        nodes = client.get_node_by_name(start, fuzzy=True)
        if not nodes:
            print(f"    未找到主体 '{start}'")
            continue
        
        found_any = False
        for node in nodes[:top_k_nodes]:
            node_id = node.get("element_id")
            if not node_id:
                continue
            cypher = """
                MATCH (n)-[r]->(m)
                WHERE elementId(n) = $nid AND type(r) = $rel
                RETURN n.name AS start, type(r) AS relation, m.name AS end
                LIMIT $limit
            """
            results = client.query(cypher, {"nid": node_id, "rel": rel, "limit": max_results_per_pattern})
            for rec in results:
                all_triples.add((rec["start"], rec["relation"], rec["end"]))
                found_any = True
        if found_any:
            print(f"    宽松匹配到 {len(all_triples)} 条三元组（累计）")
        else:
            print(f"    未找到关系 '{rel}' 从 '{start}' 出发")
    
    return list(all_triples)

In [7]:
# 单元格 7：执行全局子图匹配
if global_triple_patterns:
    print("开始匹配全局子图模式...")
    global_triples = match_global_subgraph(global_triple_patterns, client)
    print(f"\n全局检索共获得 {len(global_triples)} 条三元组。")
else:
    global_triples = []
    print("无全局子图模式，跳过匹配。")

开始匹配全局子图模式...
  模式 (陈氏太极拳)-[:展现]->(动作要素) 精确匹配失败，尝试宽松匹配...
      [Neo4j] 查找 '陈氏太极拳' (fuzzy=True) 返回 10 个节点
    未找到关系 '展现' 从 '陈氏太极拳' 出发
  模式 (陈氏太极拳)-[:传承]->(传承人) 精确匹配失败，尝试宽松匹配...
      [Neo4j] 查找 '陈氏太极拳' (fuzzy=True) 返回 10 个节点
    未找到关系 '传承' 从 '陈氏太极拳' 出发

全局检索共获得 0 条三元组。


In [8]:
# 单元格 8：合并本地与全局三元组
local_file = "data/retrieved_triples_local.json"
local_triples = []
if Path(local_file).exists():
    with open(local_file, "r", encoding="utf-8") as f:
        local_triples = json.load(f)
    print(f"已加载本地检索三元组 {len(local_triples)} 条。")
else:
    print("未找到本地检索结果文件。")

# 合并去重
combined_triples_set = set()
for t in local_triples:
    combined_triples_set.add(tuple(t) if isinstance(t, list) else t)
for t in global_triples:
    combined_triples_set.add(tuple(t) if isinstance(t, list) else t)

combined_triples = [list(t) for t in combined_triples_set]
print(f"合并后知识池共有 {len(combined_triples)} 条三元组。")

已加载本地检索三元组 168 条。
合并后知识池共有 168 条三元组。


In [9]:
# 单元格 9：保存合并结果
output_file = "data/retrieved_triples_combined.json"
Path("data").mkdir(exist_ok=True)

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(combined_triples, f, ensure_ascii=False, indent=2)

print(f"合并知识池已保存至：{output_file}")

合并知识池已保存至：data/retrieved_triples_combined.json
